In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
import win32com.client as win32
import time  # Для измерения времени выполнения
import shutil
import re
from glob import glob
import gc
from datetime import timedelta

# Функция для форматирования времени в часы, минуты и секунды
def format_elapsed_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours)} часа(ов) {int(minutes)} минут(ы) {seconds:.2f} секунд"

# Функция для проверки, является ли файл скрытым (для Windows)
def is_hidden(file_path):
    try:
        # Получаем атрибуты файла
        file_attributes = os.stat(file_path).st_file_attributes
        # Проверяем, установлен ли флаг "скрытый"
        return file_attributes & 2 != 0  # 2 соответствует атрибуту "скрытый"
    except Exception:
        # Если возникла ошибка, считаем файл не скрытым
        return False

# Функция для форматирования даты в строковый формат 'YYYY-MM-DD'
def format_date_column(df, date_column):
    if date_column in df.columns:
        df[date_column] = pd.to_datetime(df[date_column], errors='coerce').dt.strftime('%Y-%m-%d')
    return df

# Функция для подключения к SQL Server с аутентификацией Windows
def connect_to_sql(server, database):
    connection_string = (
        f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server"
        "&trusted_connection=yes"
    )
    engine = create_engine(connection_string)
    return engine

# Функция для обработки ошибок и замены их на null
def handle_errors(df):
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
    return df

In [2]:
FOLDER_PATH = os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!")
FOLDER_PATH_FEATURES = r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Дашбоард по рекламным кампаниям"
FOLDER_PATH_FOR_DB= os.path.normpath(r"\\kari.local\public\all\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям")
FOLDER_PATH_DUDL = os.path.normpath(r"\\kari.local\public\all\Агрегаторы\Дашборд реклама WB_OZ")

SQL_SERVER = "cl01sql"
SQL_DATABASE_DBREPORT = "DBReport"
SQL_DATABASE_DBPARTNERS = "DBPartners"

In [3]:
print("Начинаем собирать Базу Данных...")
start_all_time = time.time()
engine = connect_to_sql(SQL_SERVER, SQL_DATABASE_DBPARTNERS)

Начинаем собирать Базу Данных...


In [4]:
# 4. Получить данные из файла "Справочник.xlsx"
try:
    print("Начинаем получать данные для Справочника...")
    start_time = time.time()  # Запускаем таймер
    file_path_reference = os.path.join(FOLDER_PATH, "Справочник.xlsx")

    if os.path.exists(file_path_reference):
        # Список столбцов, которые нужно взять из файла
        columns_to_read = [
            "Артикул", "Артикул OZ", "Наименование", "Коллекция",
            "Бренд", "Размер", "Сезон", "Направление", "Розничный отдел",
            "Модель", "Группа", "Бизнес-группа", "Техсегмент",
            "Байер", "Две последние коллекции", "Основной артикул", "Ответственный за группу", "Себестоимость с НДС",
            "Процент выкупа", "НДС", "Группа для отчетов"
        ]

        # Типы данных для столбцов
        column_dtypes = {
            "Артикул": str,
            "Артикул OZ": str,
            "Наименование": str,
            "Коллекция": str,
            "Размер": str,
            "Бренд": str,
            "Сезон": str,
            "Направление": str,
            "Розничный отдел": str,
            "Модель": str,
            "Группа": str,
            "Бизнес-группа": str,
            "Техсегмент": str,
            "Байер": str,
            "Две последние коллекции": str,
            "Основной артикул": str,
            "Ответственный за группу": str,
            "Себестоимость с НДС": float,
            "Процент выкупа": float,
            "НДС": int,
            "Группа для отчетов": str
        }

        # Чтение файла с указанием нужных столбцов и типов данных
        df_reference = pd.read_excel(
            file_path_reference,
            sheet_name="Выгрузка для справочника",
            engine="calamine",
            usecols=columns_to_read,
            dtype=column_dtypes
        )

        # Удаление дубликатов
        df_reference = df_reference.drop_duplicates()

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Справочник:")
        print(df_reference.head())

        # Сохраняем результат
        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Справочника успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Справочник.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Справочника: {e}")

Начинаем получать данные для Справочника...
Первые 5 строк таблицы Справочник:
    Артикул                Наименование Размер Коллекция       Бренд  \
0  W2358283  Балетки женские MYZ25S-108     38    2025SS  T.TACCARDI   
1  W2358283  Балетки женские MYZ25S-108     39    2025SS  T.TACCARDI   
2  W2358283  Балетки женские MYZ25S-108     40    2025SS  T.TACCARDI   
3  W2358283  Балетки женские MYZ25S-108     41    2025SS  T.TACCARDI   
4  W2358300  Балетки женские MYZ25S-268     36    2025SS  T.TACCARDI   

             Сезон    Направление Розничный отдел      Модель Бизнес-группа  \
0  лето (закрытое)  Женская обувь   Женская обувь  MYZ25S-108         Обувь   
1  лето (закрытое)  Женская обувь   Женская обувь  MYZ25S-108         Обувь   
2  лето (закрытое)  Женская обувь   Женская обувь  MYZ25S-108         Обувь   
3  лето (закрытое)  Женская обувь   Женская обувь  MYZ25S-108         Обувь   
4  лето (закрытое)  Женская обувь   Женская обувь  MYZ25S-268         Обувь   

   ... Техсег

In [5]:
# 7. Создание таблицы "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу ВсегоРазмеров...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Артикул", "Размер"]
    for col in required_columns:
        if col not in df_reference.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_reference.")
            exit()

    # Очищаем столбец "Размер":
    # - Преобразуем в строковый формат
    # - Удаляем лишние пробелы
    # - Заменяем пустые строки на None
    df_reference["Размер"] = df_reference["Размер"].astype(str).str.strip().replace('', None)

    # Создаем DataFrame с количеством размеров для каждого артикула
    df_reference_unique = (
        df_reference
        .drop_duplicates(subset=["Артикул", "Размер"])  # Удаляем дубликаты Артикул-Размер
        .groupby("Артикул")["Размер"]  # Группируем по артикулу
        .apply(lambda sizes: len(sizes.dropna().unique()) if len(sizes.dropna()) > 0 else 1)  # Подсчитываем размеры
        .reset_index(name="Всего размеров")
    )

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВсегоРазмеров:")
    print(df_reference_unique.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВсегоРазмеров успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВсегоРазмеров: {e}")

Начинаем создавать таблицу ВсегоРазмеров...
Первые 5 строк таблицы ВсегоРазмеров:
    Артикул  Всего размеров
0  00001851               5
1  00001852               5
2  00001855               5
3  00001856               5
4  00001931               6
Таблица ВсегоРазмеров успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 30.59 секунд


In [6]:
# 6. Получить данные таблицы с SQL (РазмерыНаАгрегаторе)
try:
    print("Начинаем получать данные для РазмеровНаАгрегаторе...")
    start_time = time.time()  # Запускаем таймер
    query_sizes = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], COUNT(DISTINCT(a.[INVENTSIZEID])) AS [Колво размеров]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_sizes = pd.read_sql(query_sizes, engine)
    df_sizes = format_date_column(df_sizes, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы РазмерыНаАгрегаторе:")
    print(df_sizes.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для РазмеровНаАгрегаторе: {e}")

Начинаем получать данные для РазмеровНаАгрегаторе...
Первые 5 строк таблицы РазмерыНаАгрегаторе:
         Дата   Артикул  Колво размеров
0  2025-07-17  00006170               1
1  2025-07-17  00006410               4
2  2025-07-17  00006630               1
3  2025-07-17  00006730               1
4  2025-07-17  00128845               4
Данные для РазмеровНаАгрегаторе успешно сохранены. Время выполнения: 0 часа(ов) 3 минут(ы) 43.82 секунд


In [7]:
# 8. Связать "РазмерыНаАгрегаторе" с "ВсегоРазмеров"
try:
    print("Начинаем создавать таблицу Дистрибуция...")
    start_time = time.time()  # Запускаем таймер

    # Объединяем таблицы по полю "Артикул"
    df_distribution = pd.merge(df_sizes, df_reference_unique, on="Артикул", how="left")

    # Вычисляем дистрибуцию с проверкой на деление на ноль
    df_distribution["Дистрибуция"] = df_distribution.apply(
        lambda row: row["Колво размеров"] / row["Всего размеров"] if row["Всего размеров"] != 0 else 0,
        axis=1
    )

    # Оставляем только нужные столбцы
    df_distribution = df_distribution[["Дата", "Артикул", "Дистрибуция"]]

    # Форматирование даты
    df_distribution = format_date_column(df_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Дистрибуция:")
    print(df_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Дистрибуция успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Дистрибуция: {e}")

Начинаем создавать таблицу Дистрибуция...
Первые 5 строк таблицы Дистрибуция:
         Дата   Артикул  Дистрибуция
0  2025-07-17  00006170     0.166667
1  2025-07-17  00006410     0.666667
2  2025-07-17  00006630     0.166667
3  2025-07-17  00006730     0.166667
4  2025-07-17  00128845     1.000000
Таблица Дистрибуция успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 30.50 секунд


In [8]:
# 2. Получить данные таблицы с SQL (Остатки)
try:
    print("Начинаем получать данные для Остатков...")
    start_time = time.time()  # Запускаем таймер
    query_stock = f"""
        SELECT a.[dt] AS [Дата], a.[itemid] AS [Артикул], SUM(a.[free_to_sell_amount]) AS [Остаток Агрегатора]
        FROM [DBPartners].[dbo].[WblmRepGetStockOzon] a
        WHERE [dt] >= '{(pd.Timestamp.today() - pd.DateOffset(months=3)).strftime("%Y-%m-%d")}'
        GROUP BY [dt], [itemid]
    """
    df_stock = pd.read_sql(query_stock, engine)
    df_stock = format_date_column(df_stock, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатков:")
    print(df_stock.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Данные для Остатков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при получении данных для Остатков: {e}")

Начинаем получать данные для Остатков...
Первые 5 строк таблицы Остатков:
         Дата   Артикул  Остаток Агрегатора
0  2025-07-17  00006170                   0
1  2025-07-17  00006410                   1
2  2025-07-17  00006630                   0
3  2025-07-17  00006730                   0
4  2025-07-17  00128845                 104
Данные для Остатков успешно сохранены. Время выполнения: 0 часа(ов) 0 минут(ы) 43.43 секунд


In [9]:
# 9. Связать "Остатки" с "Дистрибуция"
try:
    print("Начинаем создавать таблицу Остатки с дистрибуцией...")
    start_time = time.time()  # Запускаем таймер
    df_stock_with_distribution = pd.merge(df_stock, df_distribution, on=["Дата", "Артикул"], how="left")
    df_stock_with_distribution = format_date_column(df_stock_with_distribution, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы Остатки с дистрибуцией:")
    print(df_stock_with_distribution.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица Остатки с дистрибуцией успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы Остатки с дистрибуцией: {e}")

Начинаем создавать таблицу Остатки с дистрибуцией...
Первые 5 строк таблицы Остатки с дистрибуцией:
         Дата   Артикул  Остаток Агрегатора  Дистрибуция
0  2025-07-17  00006170                   0     0.166667
1  2025-07-17  00006410                   1     0.666667
2  2025-07-17  00006630                   0     0.166667
3  2025-07-17  00006730                   0     0.166667
4  2025-07-17  00128845                 104     1.000000
Таблица Остатки с дистрибуцией успешно создана. Время выполнения: 0 часа(ов) 0 минут(ы) 4.11 секунд


In [10]:
del df_stock

In [11]:
import pandas as pd
import glob
import os
import datetime
import pyodbc

# Папки
path_voronka = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\ВЫГРУЗКА воронка Озон"
path_zatraty = r"Z:\Analytics\Marketplaceanalytics\Федоров\Дашбоард по рекламным кампаниям\!!!_ИСХОДНИКИ ДЛЯ ДАШБОРДА_НЕ УДАЛЯТЬ_!!!\Затраты\Озон. Затраты из Аналитики"

# === 1. ВОРОНКА ===
df_voronka_list = []
files_voronka = glob.glob(os.path.join(path_voronka, "analytics_report_*.xlsx"))

for file in files_voronka:
    # достаём дату из имени файла
    fname = os.path.basename(file)
    try:
        report_date = datetime.datetime.strptime(fname.split("_")[2], "%Y-%m-%d").date() - datetime.timedelta(days=1)
    except Exception:
        continue

    df = pd.read_excel(file, engine='calamine')

    # Чистим "Позиция в поиске и каталоге" от запятых
    df["Позиция в поиске и каталоге"] = (
        df["Позиция в поиске и каталоге"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

    # df["Позиция в поиске и каталоге"] = pd.to_numeric(df["Позиция в поиске и каталоге"], errors="coerce")
    # print(df.head(20))
    # типы
    df = df.astype({
        "Артикул": "string",
        "Показы, всего": "Int64",
        "Показы на карточке товара": "Int64",
        "Показы в поиске и каталоге": "Int64",
        "Позиция в поиске и каталоге": "float64",
        "В корзину, всего": "Int64",
        "Заказано товаров": "Int64",
        "Отменено товаров": "Int64",
        "Доставлено товаров": "Int64",
        "Возвращено товаров": "Int64",
        "Заказано на сумму": "float64",
        "В корзину из карточки товара": "Int64"
    })

    df["Дата"] = report_date
    df["Выкупили ШТ"] = df["Заказано товаров"] - df["Отменено товаров"] - df["Возвращено товаров"]
    df["Артикул"] = df["Артикул"].astype(str).str.split("-").str[0]

    df_voronka_list.append(df)

In [12]:
df_voronka = pd.concat(df_voronka_list, ignore_index=True)

In [53]:
# === 2. ЗАТРАТЫ ===
df_zatraty_list = []
files_zatraty = glob.glob(os.path.join(path_zatraty, "*.csv"))

for f in files_zatraty:
    # дата из имени файла
    fname = os.path.basename(f).replace(".csv", "")
    file_date = pd.to_datetime(fname, dayfirst=True, errors="coerce")

    # читаем csv
    df = pd.read_csv(f, sep=";", skiprows=2)

    # чистим названия колонок
    df.columns = [c.replace(".csv","") if ".csv" in c else c for c in df.columns]

    # вставляем дату из имени файла
    df["Дата"] = file_date

    # переименования
    df = df.rename(columns={
        "Тип продвижения": "ТипАктивности",
        "Расход, ₽, с НДС": "Расход, ₽"
    })

    df_zatraty_list.append(df)

df_zatraty = pd.concat(df_zatraty_list, ignore_index=True)

In [14]:
# === 3. SQL ЦЕНЫ ===
sql = """
select 'OZ' as AGREGATOR, DT, ITEMID, PRICE
from [DBPartners].[dbo].[WblmRepPriceDiscountOzReport]
where dt >= '2025-08-01'
"""
df_prices = pd.read_sql(sql, engine)

df_prices = df_prices.groupby(["DT", "ITEMID"], as_index=False).agg({"PRICE": "max"})
df_prices = df_prices.rename(columns={"DT": "Дата", "ITEMID": "Артикул", "PRICE": "Цена"})


In [15]:
df_reference = df_reference.drop_duplicates(subset=["Артикул"])
df_reference = df_reference[["Артикул", "Бизнес-группа", "Направление", "Розничный отдел", "Группа", "Модель", "Бренд", "Коллекция", "Сезон", "Себестоимость с НДС", "Процент выкупа", "Две последние коллекции", 'Артикул OZ', 'Наименование', 'Техсегмент', 'Байер', 'Основной артикул', 'НДС', 'Ответственный за группу', 'Группа для отчетов']]

In [16]:
df_reference

,Артикул,Бизнес-группа,Направление,Розничный отдел,Группа,Модель,Бренд,Коллекция,Сезон,Себестоимость с НДС,Процент выкупа,Две последние коллекции,Артикул OZ,Наименование,Техсегмент,Байер,Основной артикул,НДС,Ответственный за группу,Группа для отчетов
0,W2358283,Обувь,Женская обувь,Женская обувь,000 Балетки женские,MYZ25S-108,T.TACCARDI,2025SS,лето (закрытое),703.8850,0.937500,2025SS,2222638871,Балетки женские MYZ25S-108,flat (AM),Коновалова А.,W2358283,20,Данилов Артем,Обувь
4,W2358300,Обувь,Женская обувь,Женская обувь,000 Балетки женские,MYZ25S-268,T.TACCARDI,2025SS,лето (закрытое),719.2084,1.000000,2025SS,1857547621,Балетки женские MYZ25S-268,flat (AM),Коновалова А.,W2358300,20,Данилов Артем,Обувь
10,W2358369,Обувь,Женская обувь,Женская обувь,000 Балетки женские,ZS25S-36,T.TACCARDI,2025SS,лето (закрытое),575.8311,0.973684,2025SS,2310608969,Балетки женские ZS25S-36,flat (AM),Коновалова А.,W2358369,20,Данилов Артем,Обувь
16,W2308001,Обувь,Женская обувь,Женская обувь,000 Балетки женские,3590-103DB,T.TACCARDI,2020SS,лето (закрытое),425.1829,0.952667,2020SS,170334202,Балетки женские 3590-103DB,flat (L),Коновалова А.,W2308001,20,Данилов Артем,Обувь
22,W2350005,Обувь,Женская обувь,Женская обувь,000 Балетки женские,ZS21S-41A,T.TACCARDI,2023SS,лето (закрытое),443.4429,0.833333,"2022SS,2023SS",225123717,Балетки женские ZS21S-41A,flat (AM),Коновалова А.,W2350005,20,Данилов Артем,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
666029,25001000,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K7355-WD2,KariKids,2021AW,всесезонный,NaN,NaN,2021AW,NaN,Воздушные шары KariKids бел/син K7355-WD2,NaN,NaN,25001000,20,х,NaN
666030,25008030,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6805,KariKids,2021SS,всесезонный,NaN,NaN,2021SS,NaN,Воздушные шары KARIKIDS бел. K6805,NaN,NaN,25008030,20,х,NaN
666031,25008010,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6803,kari,2021AW,всесезонный,NaN,NaN,2021AW,NaN,Воздушные шары KARI фиолет. K6803,NaN,NaN,25008010,20,х,NaN
666032,25008040,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6806,KariKids,2021SS,всесезонный,NaN,NaN,2021SS,NaN,Воздушные шары KARIKIDS фиолет. K6806,NaN,NaN,25008040,20,х,NaN


In [ ]:
# df_voronka.drop_duplicates(subset=['Артикул', 'Дата'], inplace=True)

In [113]:
# === 5. СЛИЯНИЯ ===
df = df_voronka
df['Дата'] = pd.to_datetime(df['Дата'])
df_zatraty['SKU'] = df_zatraty['SKU'].astype(str)

# merge со справочником
df = df.merge(df_reference, on="Артикул", how="left")

# merge с затратами
df = df.merge(df_zatraty, left_on=["Дата","Артикул OZ"], right_on=["Дата","SKU"], how="left")
df_prices['Дата'] = pd.to_datetime(df_prices['Дата'])
# merge с ценами
df = df.merge(df_prices, on=["Дата","Артикул"], how="left")

In [114]:
# === 4. Справочник ===
# Заглушка: здесь нужно подгрузить df_spravochnik (например, Excel или SQL)
# df_spravochnik = pd.read_excel("справочник.xlsx")

# === 6. ДОП. ЛОГИКА ===
# def calc_status(row):
#     if row.get("Бизнес-группа") == "Не одежда/обувь":
#         return "Не одежда/обувь"
#     val = row.get("Две последние коллекции")
#     if pd.isna(val):
#         return "Без коллекции"
#     if "2024AW" not in str(val) and "2025" not in str(val):
#         return "Старая"
#     if str(val) == "2024AW" or "2025" in str(val):
#         return "Новинка без повтора"
#     return "Новинка с повтором"

# df["СтатусКоллекции"] = df.apply(calc_status, axis=1)
# df["Вид товара"] = df["СтатусКоллекции"].apply(lambda x: "новинка" if "Новинка" in str(x) else "сток")

# === 7. ВЫВОД ===
final_columns = [
    "Дата","Артикул","Показы, всего",
    "Показы на карточке товара","Показы в поиске и каталоге","Позиция в поиске и каталоге",
    "В корзину, всего","Заказано товаров","Отменено товаров","Доставлено товаров",
    "Возвращено товаров","Заказано на сумму","В корзину из карточки товара","Выкупили ШТ",
    "ТипАктивности","Расход, ₽","Продажи, ₽","Заказы, шт","Показы","Клики",
    "СтатусКоллекции","Вид товара","Цена"
]


# некоторые колонки могут отсутствовать (если справочник не подгрузили), поэтому фильтруем по пересечению
df_funnel = df[[c for c in final_columns if c in df.columns]]

# Переименование столбцов
df_funnel.rename(columns={
    "Продажи, ₽": "Рекламные заказано на сумму",
    "Показы": "Рекламные показы",
    "Клики": "Рекламные показы на карточке товара",
    "Заказы, шт": "Рекламные заказано товаров"
}, inplace=True, errors="ignore")

# df_funnel.to_excel("ozon_funnel_report_all.xlsx", index=False)

C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_17468\887558485.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_funnel.rename(columns={


In [115]:
df_funnel['ТипАктивности'].unique()

array([nan, 'Трафареты', 'Оплата за заказ', 'Спецразмещение',
       'Вывод в топ', 'Оплата за клик'], dtype=object)

In [116]:
# columns_to_read = [
#                         "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
#                         "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
#                         "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
#                         "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Продажи, ₽", "Заказы, шт", "Показы", "Клики",  "Цена"
#                     ]

In [117]:
# df_funnel = df_funnel[columns_to_read]

In [118]:
# # 3. Получить данные из файлов вложенной папки "Показатели по дням"
# try:
#     print("Начинаем получать данные для Воронки...")
#     start_time = time.time()  # Запускаем таймер
#     folder_path_weeks = os.path.join(FOLDER_PATH, "Показатели по дням")
#     df_funnel = pd.DataFrame()

#     if os.path.exists(folder_path_weeks):
#         for file in os.listdir(folder_path_weeks):
#             file_path = os.path.join(folder_path_weeks, file)

#             # Пропускаем скрытые файлы
#             if is_hidden(file_path):
#                 print(f"Пропущен скрытый файл: {file}")
#                 continue

#             # Проверяем расширение файла
#             if file.endswith((".xlsx", ".xls")):
#                 try:
#                     # Список столбцов, которые нужно взять из файла
#                     columns_to_read = [
#                         "Дата",	"Артикул", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
#                         "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
#                         "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
#                         "Выкупили ШТ", "ТипАктивности", "Расход, ₽", "Продажи, ₽", "Заказы, шт", "Показы", "Клики",  "Цена"
#                     ]

#                     if file.endswith(".xlsx"):
#                         temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="openpyxl", usecols=columns_to_read)
#                     elif file.endswith(".xls"):
#                         temp_df = pd.read_excel(file_path, sheet_name="Воронка", engine="xlrd", usecols=columns_to_read)

#                     # Переименование столбцов
#                     temp_df.rename(columns={
#                         "Артикул": "Артикул",
#                         "Продажи, ₽": "Рекламные заказано на сумму",
#                         "Показы": "Рекламные показы",
#                         "Клики": "Рекламные показы на карточке товара",
#                         "Заказы, шт": "Рекламные заказано товаров"
#                     }, inplace=True, errors="ignore")

#                     # Типы данных для столбцов
#                     column_dtypes = {
#                         "Артикул": str,
#                         "Показы, всего": int,
#                         "Показы на карточке товара": int,
#                         "Показы в поиске и каталоге": int,
#                         "Позиция в поиске и каталоге": float,
#                         "В корзину, всего": int,
#                         "Заказано товаров": int,
#                         "Отменено товаров": int,
#                         "Доставлено товаров": int,
#                         "Возвращено товаров": int,
#                         "Заказано на сумму": int,
#                         "Выкупили ШТ": int,
#                         "В корзину из карточки товара": int,
#                         "ТипАктивности": str,
#                         "Рекламные заказано на сумму": int,
#                         "Рекламные заказано товаров": int,
#                         "Рекламные показы на карточке товара": int,
#                         "Рекламные показы": int,
#                         "Расход, ₽": int,
#                         "Цена": int
#                     }

#                     # Форматирование даты
#                     temp_df = format_date_column(temp_df, 'Дата')

#                     df_funnel = pd.concat([df_funnel, temp_df])
#                 except Exception as e:
#                     print(f"Ошибка при чтении файла {file}: {e}")

#         # Удаление лишних столбцов (если они остались)
#         df_funnel = df_funnel[[
#              "Дата",	"Артикул", "ТипАктивности", "Показы, всего", "Показы на карточке товара", "Показы в поиске и каталоге",
#                         "Позиция в поиске и каталоге", "В корзину, всего", "Заказано товаров", "Отменено товаров",
#                         "Доставлено товаров", "Возвращено товаров", "Заказано на сумму", "В корзину из карточки товара",
#                         "Выкупили ШТ", "Расход, ₽", "Рекламные заказано на сумму", "Рекламные заказано товаров",
#                         "Рекламные показы", "Рекламные показы на карточке товара", "Цена"
#         ]]
#         mask = pd.to_numeric(df_funnel['Расход, ₽'], errors='coerce').eq(0)
#         df_funnel.loc[mask, 'ТипАктивности'] = 'Органика'

#         # Вывод первых 5 строк
#         print("Первые 5 строк таблицы Воронка:")
#         print(df_funnel.head())

#         elapsed_time = time.time() - start_time  # Вычисляем затраченное время
#         print(f"Данные для Воронки успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
#     else:
#         print("Папка 'Показатели по дням' не найдена.")
# except Exception as e:
#     print(f"Ошибка при получении данных для Воронки: {e}")

In [119]:
df_funnel.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'ТипАктивности',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'],
      dtype='object')

In [120]:
funnel_columns = df_funnel.columns

In [121]:
# Путь до папки
folder_path_weeks = os.path.join(FOLDER_PATH, "Затраты", "Озон. Затраты из Аналитики New Format")

# Собираем все .xlsx файлы
files = glob.glob(os.path.join(folder_path_weeks, "*.xlsx"))

df_list = []
for file in files:
    # --- достаём дату из названия файла ---
    filename = os.path.basename(file)  # например: "Аналитика продвижения_16.09.2025.xlsx"
    date_str = filename.split("_")[-1].replace(".xlsx", "")  # "16.09.2025"
    date_parsed = (pd.to_datetime(date_str, format="%d.%m.%Y") - timedelta(days=1)).strftime("%Y-%m-%d")

    # читаем, пропуская первую строку
    df_tmp = pd.read_excel(file, skiprows=1)

    # оставляем только нужные колонки
    cols_keep = ["SKU", "ID кампании", "Инструмент", "Место размещения"]
    df_tmp = df_tmp[cols_keep]

    # добавляем колонку "Дата"
    df_tmp["Дата"] = date_parsed

    df_list.append(df_tmp)

# объединяем все файлы
df_all = pd.concat(df_list, ignore_index=True)
df_all.rename(columns={'SKU': "Артикул OZ"},inplace=True)
df_all["Артикул OZ"] = df_all["Артикул OZ"].astype(str)

In [122]:
df_reference

,Артикул,Бизнес-группа,Направление,Розничный отдел,Группа,Модель,Бренд,Коллекция,Сезон,Себестоимость с НДС,Процент выкупа,Две последние коллекции,Артикул OZ,Наименование,Техсегмент,Байер,Основной артикул,НДС,Ответственный за группу,Группа для отчетов
0,W2358283,Обувь,Женская обувь,Женская обувь,000 Балетки женские,MYZ25S-108,T.TACCARDI,2025SS,лето (закрытое),703.8850,0.937500,2025SS,2222638871,Балетки женские MYZ25S-108,flat (AM),Коновалова А.,W2358283,20,Данилов Артем,Обувь
4,W2358300,Обувь,Женская обувь,Женская обувь,000 Балетки женские,MYZ25S-268,T.TACCARDI,2025SS,лето (закрытое),719.2084,1.000000,2025SS,1857547621,Балетки женские MYZ25S-268,flat (AM),Коновалова А.,W2358300,20,Данилов Артем,Обувь
10,W2358369,Обувь,Женская обувь,Женская обувь,000 Балетки женские,ZS25S-36,T.TACCARDI,2025SS,лето (закрытое),575.8311,0.973684,2025SS,2310608969,Балетки женские ZS25S-36,flat (AM),Коновалова А.,W2358369,20,Данилов Артем,Обувь
16,W2308001,Обувь,Женская обувь,Женская обувь,000 Балетки женские,3590-103DB,T.TACCARDI,2020SS,лето (закрытое),425.1829,0.952667,2020SS,170334202,Балетки женские 3590-103DB,flat (L),Коновалова А.,W2308001,20,Данилов Артем,Обувь
22,W2350005,Обувь,Женская обувь,Женская обувь,000 Балетки женские,ZS21S-41A,T.TACCARDI,2023SS,лето (закрытое),443.4429,0.833333,"2022SS,2023SS",225123717,Балетки женские ZS21S-41A,flat (AM),Коновалова А.,W2350005,20,Данилов Артем,Обувь
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
666029,25001000,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K7355-WD2,KariKids,2021AW,всесезонный,NaN,NaN,2021AW,NaN,Воздушные шары KariKids бел/син K7355-WD2,NaN,NaN,25001000,20,х,NaN
666030,25008030,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6805,KariKids,2021SS,всесезонный,NaN,NaN,2021SS,NaN,Воздушные шары KARIKIDS бел. K6805,NaN,NaN,25008030,20,х,NaN
666031,25008010,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6803,kari,2021AW,всесезонный,NaN,NaN,2021AW,NaN,Воздушные шары KARI фиолет. K6803,NaN,NaN,25008010,20,х,NaN
666032,25008040,Оборудование,Оборудование,Оборудование для магазина,z13 Материалы для оформления магазина,K6806,KariKids,2021SS,всесезонный,NaN,NaN,2021SS,NaN,Воздушные шары KARIKIDS фиолет. K6806,NaN,NaN,25008040,20,х,NaN


In [123]:
# 10. Связать "Воронка" с "Справочник"
try:
    print("Начинаем создавать таблицу ВоронкаСправочник...")
    start_time = time.time()  # Запускаем таймер

    # Проверка наличия необходимых столбцов
    required_columns = ["Дата", "Артикул"]
    for col in required_columns:
        if col not in df_funnel.columns:
            print(f"Ошибка: Отсутствует столбец '{col}' в df_funnel.")
            exit()

    # Список столбцов, которые нужно взять из справочника
    reference_columns = [
        "Артикул", "Артикул OZ", "Наименование", "Коллекция", "Бренд", "Сезон", "Направление",
        "Розничный отдел", "Модель", "Группа", "Бизнес-группа", "Техсегмент",
        "Байер", "Две последние коллекции", "Основной артикул", "Себестоимость с НДС",
        "Процент выкупа", "НДС", "Ответственный за группу", "Группа для отчетов"
    ]

    # Фильтруем справочник, оставляя только нужные столбцы
    df_reference_filtered = df_reference[reference_columns]

    # Приводим типы данных к строковому формату
    df_funnel["Артикул"] = df_funnel["Артикул"].fillna('').astype(str).str[:8]  # Заменяем NaN на пустые строки
    df_reference_filtered["Артикул"] = df_reference_filtered["Артикул"].fillna('').astype(str).str[:8]

    # Объединение таблиц
    df_funnel_reference = pd.merge(
        df_funnel,
        df_reference_filtered,
        left_on="Артикул",
        right_on="Артикул",
        how="left"
    )

    # Удаление дубликатов
    # df_funnel_reference = df_funnel_reference.drop_duplicates()

    # Удаление лишних столбцов (если они остались)
    # df_funnel_reference = df_funnel_reference.drop(columns=["Артикул WB"], errors="ignore")

    # Форматирование даты
    df_funnel_reference = format_date_column(df_funnel_reference, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ВоронкаСправочник:")
    print(df_funnel_reference.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ВоронкаСправочник успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ВоронкаСправочник: {e}")

Начинаем создавать таблицу ВоронкаСправочник...


C:\Users\i.taldykin\AppData\Local\Temp\ipykernel_17468\2539683674.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_funnel["Артикул"] = df_funnel["Артикул"].fillna('').astype(str).str[:8]  # Заменяем NaN на пустые строки


Первые 5 строк таблицы ВоронкаСправочник:
         Дата   Артикул  Показы, всего  Показы на карточке товара  \
0  2025-07-31  W2655789         157068                       3426   
1  2025-07-31  W5252198         110595                       1301   
2  2025-07-31  S5258430          79473                        743   
3  2025-07-31  a2205000          65630                       1049   
4  2025-07-31  a2305000          63599                        830   

   Показы в поиске и каталоге  Позиция в поиске и каталоге  В корзину, всего  \
0                          90                       215.56                43   
1                        1559                       120.70                11   
2                        2089                        66.98                 5   
3                        6529                         9.73               310   
4                        5019                        13.18               247   

   Заказано товаров  Отменено товаров  Доставлено товаров  ...

In [124]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Бизнес-группа,Техсегмент,Байер,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов
0,2025-07-31,W2655789,157068,3426,90,215.56,43,2,0,0,...,Обувь,flat (AM),Коновалова А.,2023AW,W2655789,666.6063,0.972973,20.0,Данилов Артем,Обувь
1,2025-07-31,W5252198,110595,1301,1559,120.70,11,1,0,0,...,Обувь,flat (AM),Коновалова А.,2022SS,W5252198,834.2605,1.000000,20.0,Данилов Артем,Обувь
2,2025-07-31,S5258430,79473,743,2089,66.98,5,1,1,0,...,Обувь,flat (AM),Коновалова А.,2025SS,S5258430,930.9122,1.000000,10.0,Жилин Даниил,Обувь
3,2025-07-31,a2205000,65630,1049,6529,9.73,310,391,10,125,...,Элементы питания,NaN,Зайцев И. Игрушки,"2025AW,2025SS",a2205000,34.1842,0.967172,20.0,Валиков Никита,Игрушки
4,2025-07-31,a2305000,63599,830,5019,13.18,247,205,18,109,...,Элементы питания,NaN,Зайцев И. Игрушки,"2024SS,2025SS",a2305000,40.8616,0.979489,20.0,Валиков Никита,Игрушки
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15296869,2025-10-16,y0900000,0,0,0,0.00,0,0,0,1,...,Игрушки,NaN,Зайцев И. Игрушки,"2024SS,2025AW",y0900000,208.1490,1.000000,10.0,Валиков Никита,Игрушки
15296870,2025-10-16,73007000,0,0,0,0.00,0,0,0,1,...,Товары для малышей,NaN,Бордачук Е. КГТ,"2020AW,2021SS",73007000,164.6930,0.969697,20.0,Шляпин Алексей,КГТ
15296871,2025-10-16,105051O0,0,0,0,0.00,0,0,0,0,...,Аксессуары,NaN,Коновалова А.,2023AW,105051O0,516.8839,0.952726,20.0,Харлов Эдуард,Сумки
15296872,2025-10-16,89006130,0,0,0,0.00,0,0,0,1,...,Одежда для детей,NaN,Коновалова И. Одежда,2024SS,89006130,297.7850,0.917266,10.0,Зиннуров Ильнур,Детская одежда


In [125]:
# Шаг 1. Определим, какие колонки содержат суффиксы
cols_x = [c for c in df_funnel_reference.columns if c.endswith('_x')]
cols_y = [c for c in df_funnel_reference.columns if c.endswith('_y')]

# Шаг 2. Уберем "_x" из названий колонок
rename_map = {c: c[:-2] for c in cols_x}

# Шаг 3. Выберем все нужные колонки:
# - те, у которых есть "_x"
# - все остальные, не имеющие "_y" и "_x"
cols_final = [c for c in df_funnel_reference.columns if not c.endswith('_y')]
df_funnel_reference = df_funnel_reference[cols_final].rename(columns=rename_map)

In [ ]:
# объединяем с df_funnel
df_result = df_funnel_reference.merge(
    df_all,
    on=["Дата", "Артикул OZ"],
    how="left"
)

# Трафарет
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') &
    (df_result['Место размещения'] == 'Поиск и рекомендации')) |
    (df_result['ТипАктивности'] == 'Оплата за клик')
)
df_result.loc[mask, 'ТипАктивности'] = 'Трафарет'

# Вывод в топ
mask = (
    ((df_result['Инструмент'] == 'Оплата за клик') & 
     (df_result['Место размещения'] == 'Поиск')) |
    (df_result['ТипАктивности'] == 'ТОП')
)
df_result.loc[mask, 'ТипАктивности'] = 'Вывод в топ'

# Органика
mask = (
    ((df_result['Расход, ₽'] == 0) | 
     (df_result['Расход, ₽'] == 0.0))
)
df_result.loc[mask, 'ТипАктивности'] = 'Органика'

In [127]:
df_result['ТипАктивности'].unique()

array([nan, 'Трафареты', 'Оплата за заказ', 'Спецразмещение',
       'Вывод в топ', 'Трафарет'], dtype=object)

In [128]:
# df_funnel_reference_copy = df_funnel_reference.copy()
df_funnel_reference = df_result

In [129]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 'ТипАктивности',
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена', 'Артикул OZ',
       'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление',
       'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент',
       'Байер', 'Две последние коллекции', 'Основной артикул',
       'Себестоимость с НДС', 'Процент выкупа', 'НДС',
       'Ответственный за группу', 'Группа для отчетов', 'ID кампании',
       'Инструмент', 'Место размещения'],
      dtype='object')

In [130]:
# import pandas as pd
# import numpy as np

# # --- 0) подготовка
# df = df_funnel_reference.copy()

# # переименовать тип активности "ТОП" -> "Вывод в топ"
# df.loc[df['ТипАктивности'].eq('ТОП'), 'ТипАктивности'] = 'Вывод в топ'

# # полный список типов (жёстко фиксируем порядок и наличие)
# ALL_TYPES = ['Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Спецразмещение', 'Органика']

# # ключи и метрики
# key_cols = ['Дата', 'Артикул']
# # всё, что стоит в таблице ПОСЛЕ "ТипАктивности", считаем метриками
# cols = funnel_columns
# metric_cols = cols[cols.index('ТипАктивности') + 1 :]

# print(metric_cols)

# # привести метрики к числам (на случай строк/пробелов)
# for c in metric_cols:
#     df[c] = pd.to_numeric(df[c], errors='coerce')

# # --- 1) агрегация по (Дата, Артикул, ТипАктивности)
# g = (df
#      .groupby(key_cols + ['ТипАктивности'], as_index=False)[metric_cols]
#      .sum(min_count=1)
# )

# # --- 2) «широкая» таблица метрик с префиксами <Тип>_<Метрика>
# wide_metrics = g.pivot_table(
#     index=key_cols,
#     columns='ТипАктивности',
#     values=metric_cols,
#     aggfunc='sum',
#     fill_value=0
# )

# # гарантируем наличие ВСЕХ типов и ВСЕХ метрик (даже если их не было в данных)
# full_cols = pd.MultiIndex.from_product([metric_cols, ALL_TYPES])
# wide_metrics = wide_metrics.reindex(columns=full_cols, fill_value=0)

# # имена колонок: "Тип_Метрика"
# wide_metrics.columns = [f'{act}_{met}' for met, act in wide_metrics.columns.to_flat_index()]
# wide_metrics = wide_metrics.reset_index()

# # --- 3) бинарные признаки наличия типа активности (1/0) по каждой паре (Дата, Артикул)
# presence = (
#     df.groupby(key_cols + ['ТипАктивности']).size()
#       .reset_index(name='n')
#       .pivot(index=key_cols, columns='ТипАктивности', values='n')
#       .reindex(columns=ALL_TYPES, fill_value=0)
#       .gt(0).astype(int)  # 1 если был хотя бы один ряд данного типа
#       .reset_index()
# )

# # --- 4) объединяем метрики и бинарные признаки
# out = (wide_metrics
#        .merge(presence, on=key_cols, how='left')
#        .fillna(0)
# )

# # --- 5) итоговые столбцы БЕЗ префиксов = сумма по всем типам
# for met in metric_cols:
#     to_sum = [f'{t}_{met}' for t in ALL_TYPES if f'{t}_{met}' in out.columns]
#     if to_sum:
#         out[met] = out[to_sum].sum(axis=1)

# # --- 6) порядок колонок: ключи → бинарные типы → для каждой метрики столбцы по типам → итог по метрике
# ordered = key_cols + ALL_TYPES[:]  # бинарные столбцы имеют те же имена, что и типы
# for met in metric_cols:
#     ordered += [f'{t}_{met}' for t in ALL_TYPES]
#     ordered += [met]
# # оставим только реально существующие (вдруг каких-то метрик не было)
# ordered = [c for c in ordered if c in out.columns]

# out = out[ordered]

# # результат в переменной `out`
# # одна строка на (Дата, Артикул), колоноки вида:
# # Дата | Артикул | Вывод в топ | Трафарет | ... | Вывод в топ_Показы, всего | Трафарет_Показы, всего | ... | Показы, всего | ...


In [131]:
df_funnel_reference

,Дата,Артикул,"Показы, всего",Показы на карточке товара,Показы в поиске и каталоге,Позиция в поиске и каталоге,"В корзину, всего",Заказано товаров,Отменено товаров,Доставлено товаров,...,Две последние коллекции,Основной артикул,Себестоимость с НДС,Процент выкупа,НДС,Ответственный за группу,Группа для отчетов,ID кампании,Инструмент,Место размещения
0,2025-07-31,W2655789,157068,3426,90,215.56,43,2,0,0,...,2023AW,W2655789,666.6063,0.972973,20.0,Данилов Артем,Обувь,NaN,NaN,NaN
1,2025-07-31,W5252198,110595,1301,1559,120.70,11,1,0,0,...,2022SS,W5252198,834.2605,1.000000,20.0,Данилов Артем,Обувь,NaN,NaN,NaN
2,2025-07-31,S5258430,79473,743,2089,66.98,5,1,1,0,...,2025SS,S5258430,930.9122,1.000000,10.0,Жилин Даниил,Обувь,NaN,NaN,NaN
3,2025-07-31,a2205000,65630,1049,6529,9.73,310,391,10,125,...,"2025AW,2025SS",a2205000,34.1842,0.967172,20.0,Валиков Никита,Игрушки,NaN,NaN,NaN
4,2025-07-31,a2305000,63599,830,5019,13.18,247,205,18,109,...,"2024SS,2025SS",a2305000,40.8616,0.979489,20.0,Валиков Никита,Игрушки,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15465801,2025-10-16,y0900000,0,0,0,0.00,0,0,0,1,...,"2024SS,2025AW",y0900000,208.1490,1.000000,10.0,Валиков Никита,Игрушки,NaN,NaN,NaN
15465802,2025-10-16,73007000,0,0,0,0.00,0,0,0,1,...,"2020AW,2021SS",73007000,164.6930,0.969697,20.0,Шляпин Алексей,КГТ,NaN,NaN,NaN
15465803,2025-10-16,105051O0,0,0,0,0.00,0,0,0,0,...,2023AW,105051O0,516.8839,0.952726,20.0,Харлов Эдуард,Сумки,NaN,NaN,NaN
15465804,2025-10-16,89006130,0,0,0,0.00,0,0,0,1,...,2024SS,89006130,297.7850,0.917266,10.0,Зиннуров Ильнур,Детская одежда,NaN,NaN,NaN


In [132]:
# df_funnel_reference.drop_duplicates(subset=['Артикул OZ', 'Дата'], inplace=True)

In [133]:
funnel_columns_all = [
    'Дата', 'Артикул', 'ТипАктивности', 'Показы, всего',
    'Показы на карточке товара', 'Показы в поиске и каталоге',
    'Позиция в поиске и каталоге', 'В корзину, всего',
    'Заказано товаров', 'Отменено товаров', 'Доставлено товаров',
    'Возвращено товаров', 'Заказано на сумму',
    'В корзину из карточки товара', 'Выкупили ШТ',
    'Расход, ₽', 'Рекламные заказано на сумму',
    'Рекламные заказано товаров', 'Рекламные показы',
    'Рекламные показы на карточке товара', 'Цена',
    'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон',
    'Направление', 'Розничный отдел', 'Модель', 'Группа',
    'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции',
    'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС',
    'Ответственный за группу', 'Группа для отчетов',
    'ID кампании', 'Инструмент', 'Место размещения'
]
funnel_columns_widing = ['Дата', 'Артикул', 'ТипАктивности', 'Показы, всего', 'Показы на карточке товара',
       'Показы в поиске и каталоге', 'Позиция в поиске и каталоге',
       'В корзину, всего', 'Заказано товаров', 'Отменено товаров',
       'Доставлено товаров', 'Возвращено товаров', 'Заказано на сумму',
       'В корзину из карточки товара', 'Выкупили ШТ', 
       'Расход, ₽', 'Рекламные заказано на сумму',
       'Рекламные заказано товаров', 'Рекламные показы',
       'Рекламные показы на карточке товара', 'Цена'
       ]

In [134]:
df_funnel_reference = df_funnel_reference[funnel_columns_all]

In [138]:
df_funnel_reference['ТипАктивности'].unique()

array([nan, 'Трафареты', 'Оплата за заказ', 'Спецразмещение',
       'Вывод в топ', 'Трафарет'], dtype=object)

In [135]:
df_funnel_reference.head(20).to_excel('Voronka2.xlsx')

In [136]:
import pandas as pd
import numpy as np

def build_funnel_wide(
    df_raw: pd.DataFrame,
    funnel_columns: list,
    all_types=('Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика'),
    infer_organic_by_zero_spend=False,
    spend_col='Расход, ₽'
):
    """
    Склеивает строки по (Дата, Артикул), раскладывает метрики по типам активности,
    добавляет итоги и ПРОНОСИТ все прочие (не суммируемые) колонки из df_raw.
    """

    # 0) берём только нужные колонки для расчёта метрик (экономим память/время)
    cols_present = [c for c in funnel_columns if c in df_raw.columns]
    df = df_raw.loc[:, cols_present].copy()

    # 1) нормализуем типы активности
    type_col = 'ТипАктивности'
    df[type_col] = df[type_col].replace({'ТОП': 'Вывод в топ'})

    if infer_organic_by_zero_spend and spend_col in df.columns:
        m = pd.to_numeric(df[spend_col], errors='coerce').fillna(0).eq(0)
        df.loc[m, type_col] = 'Органика'

    # ключи и метрики
    key_cols = ['Дата', 'Артикул']
    met_start = funnel_columns.index(type_col) + 1
    metric_cols = [c for c in funnel_columns[met_start:] if c in df.columns]

    # привести метрики к числам (экономный тип)
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')

    # 2) суммируем по (Дата, Артикул, ТипАктивности)
    g = df.groupby(key_cols + [type_col], as_index=False)[metric_cols].sum(min_count=1)

    # база ключей (одна строка на Дата+Артикул)
    base = g[key_cols].drop_duplicates().set_index(key_cols).sort_index()

    # --- блоки метрик по каждому типу + базовые бинарные флаги ---
    metric_blocks, flag_blocks = [], []

    for t in all_types:
        sub = g[g[type_col] == t].set_index(key_cols)

        # метрики с префиксом
        if sub.empty:
            sub_metrics = pd.DataFrame(
                0.0, index=base.index,
                columns=[f'{t}_{m}' for m in metric_cols],
                dtype='float32'
            )
        else:
            sub_metrics = (sub[metric_cols]
                           .rename(columns={m: f'{t}_{m}' for m in metric_cols})
                           .reindex(base.index, fill_value=0.0)
                           .astype('float32'))
        metric_blocks.append(sub_metrics)

        # бинарный флаг наличия типа
        if sub.empty:
            flag = pd.Series(0, index=base.index, name=t, dtype='int8')
        else:
            flag = (pd.Series(1, index=sub.index, name=t)
                      .reindex(base.index, fill_value=0)
                      .astype('int8'))
        flag_blocks.append(flag)

    metrics_block = pd.concat(metric_blocks, axis=1)
    flags_block   = pd.concat(flag_blocks, axis=1)

    # --- 3) ДОП. КОЛОНКИ (все из df_raw, которых нет в funnel_columns) ---
    #    агрегируем по (Дата, Артикул) → first (первое ненулевое значение)
    extra_cols = [c for c in df_raw.columns
                  if c not in set(key_cols + [type_col] + metric_cols)]
    if extra_cols:
        # оставим только реально существующие
        extra_cols = [c for c in extra_cols if c in df_raw.columns]
        dims_block = (df_raw[key_cols + extra_cols]
                        .sort_values(key_cols)
                        .groupby(key_cols, as_index=False)
                        .first())  # берёт первое НЕ NaN
        # переиндексируем к базе
        dims_block = (dims_block.set_index(key_cols)
                                   .reindex(base.index)
                                   .reset_index())
    else:
        dims_block = base.reset_index()

    # --- 4) итоги без префиксов (сумма по всем типам) ---
    totals = pd.DataFrame(index=base.index)
    for m in metric_cols:
        totals[m] = metrics_block[[f'{t}_{m}' for t in all_types]].sum(axis=1).astype('float32')

    # --- 5) финальная сборка ---
    out = pd.concat(
        [
            dims_block,  # ключи + все доп. колонки
            flags_block.reset_index(drop=True),
            metrics_block.reset_index(drop=True),
            totals.reset_index(drop=True)
        ],
        axis=1
    ).copy()

    # --- 6) порядок колонок: ключи → доп.колонки → флаги → метрики по типам → итоги ---
    ordered = []
    # ключи
    ordered += key_cols
    # доп. колонки в исходном порядке df_raw (после ключей)
    ordered += [c for c in df_raw.columns
                if c in out.columns and c not in key_cols + [type_col] + metric_cols]
    # флаги типов
    ordered += [t for t in all_types if t in out.columns]
    # метрики по типам + итог без префикса
    for m in metric_cols:
        ordered += [f'{t}_{m}' for t in all_types if f'{t}_{m}' in out.columns]
        if m in out.columns:
            ordered += [m]

    out = out[[c for c in ordered if c in out.columns]]

    return out

In [137]:
out = build_funnel_wide(df_raw=df_funnel_reference, funnel_columns=funnel_columns_widing)
out

,Дата,Артикул,Артикул OZ,Наименование,Коллекция,Бренд,Сезон,Направление,Розничный отдел,Модель,...,Вывод в топ_Рекламные показы на карточке товара,Трафарет_Рекламные показы на карточке товара,Оплата за заказ_Рекламные показы на карточке товара,Органика_Рекламные показы на карточке товара,Рекламные показы на карточке товара,Вывод в топ_Цена,Трафарет_Цена,Оплата за заказ_Цена,Органика_Цена,Цена
0,2025-08-01,00206040,149484634,Кеды женские летние ZY16S03A18K,2024SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZY16S03A18K,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-08-01,00206110,149484602,Кеды женские летние ZY19SS-12,2024SS,T.TACCARDI,лето (закрытое),Женская обувь,Женская обувь,ZY19SS-12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-08-01,00708200,170983395,Полуботинки женские для активного отдыха K1758...,2025SS,TimeJump,демисезонный (низкий),Женская обувь,Женская обувь,K1758-5DK,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-08-01,00708210,170983396,Полуботинки женские для активного отдыха K1758...,2025SS,TimeJump,демисезонный (низкий),Женская обувь,Женская обувь,K1758-5BK,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-08-01,00708270,174555458,Полуботинки женские для активного отдыха K1731...,2024AW,T.TACCARDI,демисезонный (низкий),Женская обувь,Женская обувь,K1731-9EK,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526565,2025-10-16,y9709010,2804437674,Футболка мужская A89508-2,2025AW,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A89508-2,...,0.0,0.0,0.0,0.0,0.0,0.0,7816.0,0.0,0.0,7816.0
526566,2025-10-16,y9709020,2804437223,Футболка мужская A89508-3,2025AW,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A89508-3,...,0.0,0.0,0.0,0.0,0.0,0.0,7816.0,0.0,0.0,7816.0
526567,2025-10-16,y9709030,2804437573,Футболка мужская A89509-1,2025AW,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A89509-1,...,0.0,0.0,0.0,0.0,0.0,0.0,7816.0,0.0,0.0,7816.0
526568,2025-10-16,y9709040,2804437112,Футболка мужская A89509-2,2025AW,kari,всесезонный,Одежда для мужчин,Спортивная одежда для мужчин,A89509-2,...,0.0,8.0,0.0,0.0,8.0,0.0,6436.0,0.0,0.0,6436.0


In [112]:
print(list(out.columns))

['Дата', 'Артикул', 'Артикул OZ', 'Наименование', 'Коллекция', 'Бренд', 'Сезон', 'Направление', 'Розничный отдел', 'Модель', 'Группа', 'Бизнес-группа', 'Техсегмент', 'Байер', 'Две последние коллекции', 'Основной артикул', 'Себестоимость с НДС', 'Процент выкупа', 'НДС', 'Ответственный за группу', 'Группа для отчетов', 'ID кампании', 'Инструмент', 'Место размещения', 'Вывод в топ', 'Трафарет', 'Оплата за заказ', 'Органика', 'Вывод в топ_Показы, всего', 'Трафарет_Показы, всего', 'Оплата за заказ_Показы, всего', 'Органика_Показы, всего', 'Показы, всего', 'Вывод в топ_Показы на карточке товара', 'Трафарет_Показы на карточке товара', 'Оплата за заказ_Показы на карточке товара', 'Органика_Показы на карточке товара', 'Показы на карточке товара', 'Вывод в топ_Показы в поиске и каталоге', 'Трафарет_Показы в поиске и каталоге', 'Оплата за заказ_Показы в поиске и каталоге', 'Органика_Показы в поиске и каталоге', 'Показы в поиске и каталоге', 'Вывод в топ_Позиция в поиске и каталоге', 'Трафарет_Поз

In [41]:
df_funnel_reference = out

In [42]:
df_funnel_reference.columns

Index(['Дата', 'Артикул', 'Две последние коллекции_x', 'Сезон_x',
       'Коллекция_x', 'Бренд_x', 'Группа_x', 'Розничный отдел_x',
       'Направление_x', 'Бизнес-группа_x',
       ...
       'Вывод в топ_Расход, ₽', 'Трафарет_Расход, ₽',
       'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽',
       'Вывод в топ_Цена', 'Трафарет_Цена', 'Оплата за заказ_Цена',
       'Органика_Цена', 'Цена'],
      dtype='object', length=115)

In [43]:
del df_funnel

In [44]:
# 11. Связать "ВоронкаСправочник" с "Остатки с дистрибуцией"
try:
    print("Начинаем создавать таблицу ДБбезПризнаков...")
    start_time = time.time()  # Запускаем таймер
    df_final_db = pd.merge(df_funnel_reference, df_stock_with_distribution, left_on=["Дата", "Артикул"], right_on=["Дата", "Артикул"], how="left")
    df_final_db = format_date_column(df_final_db, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБбезПризнаков:")
    print(df_final_db.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБбезПризнаков успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБбезПризнаков: {e}")

Начинаем создавать таблицу ДБбезПризнаков...
Первые 5 строк таблицы ДБбезПризнаков:
         Дата   Артикул Две последние коллекции_x                Сезон_x  \
0  2025-09-03  00206040             2023SS,2024SS        лето (закрытое)   
1  2025-09-03  00206110             2023SS,2024SS        лето (закрытое)   
2  2025-09-03  00708270             2023AW,2024AW  демисезонный (низкий)   
3  2025-09-03  01908120             2022SS,2023SS        лето (открытое)   
4  2025-09-03  02908010             2024SS,2025SS  демисезонный (низкий)   

  Коллекция_x                Бренд_x  \
0      2024SS             T.TACCARDI   
1      2024SS             T.TACCARDI   
2      2024AW             T.TACCARDI   
3      2023SS  Alessio Nesca Comfort   
4      2025SS               TimeJump   

                                       Группа_x Розничный отдел_x  \
0                       002 Кеды женские летние     Женская обувь   
1                       002 Кеды женские летние     Женская обувь   
2  007 Полу

In [45]:
df_final_db.columns

Index(['Дата', 'Артикул', 'Две последние коллекции_x', 'Сезон_x',
       'Коллекция_x', 'Бренд_x', 'Группа_x', 'Розничный отдел_x',
       'Направление_x', 'Бизнес-группа_x',
       ...
       'Оплата за заказ_Расход, ₽', 'Органика_Расход, ₽', 'Расход, ₽',
       'Вывод в топ_Цена', 'Трафарет_Цена', 'Оплата за заказ_Цена',
       'Органика_Цена', 'Цена', 'Остаток Агрегатора', 'Дистрибуция'],
      dtype='object', length=117)

In [46]:
# 5. Получить данные из файла !!!_Признаки для артикула и даты для Озон
try:
    print("Начинаем получать данные для Признаков...")
    start_time = time.time()  # Запускаем таймер
    file_path_features = os.path.join(FOLDER_PATH_FEATURES, "!!!_Признаки для артикула и даты для Озон.xlsx")
    if os.path.exists(file_path_features):
        df_item_features = pd.read_excel(file_path_features, sheet_name="Признаки для артикула", dtype=str, engine="openpyxl")
        df_date_features = pd.read_excel(file_path_features, sheet_name="Признаки для дат", dtype={0: "datetime64[ns]", **{i: str for i in range(1, 6)}}, engine="openpyxl")

        # Обработка ошибок
        df_item_features = handle_errors(df_item_features)
        df_date_features = handle_errors(df_date_features)

        # Форматирование даты
        df_date_features = format_date_column(df_date_features, 'Дата')

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки артикула:")
        print(df_item_features.head())

        # Вывод первых 5 строк
        print("Первые 5 строк таблицы Признаки дат:")
        print(df_date_features.head())

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Данные для Признаков успешно сохранены. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл '!!!_Признаки для артикула и даты для Озон.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при получении данных для Признаков: {e}")

Начинаем получать данные для Признаков...
Первые 5 строк таблицы Признаки артикула:
    Артикул Признак Артикула 1 Признак Артикула 2 Признак Артикула 3  \
0  00001851                NaN                NaN                NaN   
1  00001852                NaN                NaN                NaN   
2  00001855                NaN                NaN                NaN   
3  00001856                NaN                NaN                NaN   
4  00001931                NaN                NaN                NaN   

  Признак Артикула 4 Признак Артикула 5  
0                NaN                NaN  
1                NaN                NaN  
2                NaN                NaN  
3                NaN                NaN  
4                NaN                NaN  
Первые 5 строк таблицы Признаки дат:
Empty DataFrame
Columns: [Дата, Признак Даты 1, Признак Даты 2, Признак Даты 3, Признак Даты 4, Признак Даты 5]
Index: []
Данные для Признаков успешно сохранены. Время выполнения: 0 часа(ов) 0 м

In [47]:
# 12. Связать "ДБбезПризнаков" с "Признаки для артикула"
try:
    print("Начинаем создавать таблицу ДБсПризнакамиАртикула...")
    start_time = time.time()  # Запускаем таймер
    df_final_db_item_features = pd.merge(df_final_db, df_item_features, left_on=["Артикул"], right_on=["Артикул"], how="left")
    df_final_db_item_features = format_date_column(df_final_db_item_features, 'Дата')

    # Вывод первых 5 строк
    print("Первые 5 строк таблицы ДБсПризнакамиАртикула:")
    print(df_final_db_item_features.head())

    elapsed_time = time.time() - start_time  # Вычисляем затраченное время
    print(f"Таблица ДБсПризнакамиАртикула успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнакамиАртикула: {e}")

Начинаем создавать таблицу ДБсПризнакамиАртикула...
Первые 5 строк таблицы ДБсПризнакамиАртикула:
         Дата   Артикул Две последние коллекции_x                Сезон_x  \
0  2025-09-03  00206040             2023SS,2024SS        лето (закрытое)   
1  2025-09-03  00206110             2023SS,2024SS        лето (закрытое)   
2  2025-09-03  00708270             2023AW,2024AW  демисезонный (низкий)   
3  2025-09-03  01908120             2022SS,2023SS        лето (открытое)   
4  2025-09-03  02908010             2024SS,2025SS  демисезонный (низкий)   

  Коллекция_x                Бренд_x  \
0      2024SS             T.TACCARDI   
1      2024SS             T.TACCARDI   
2      2024AW             T.TACCARDI   
3      2023SS  Alessio Nesca Comfort   
4      2025SS               TimeJump   

                                       Группа_x Розничный отдел_x  \
0                       002 Кеды женские летние     Женская обувь   
1                       002 Кеды женские летние     Женская обувь 

In [48]:
del df_item_features

In [49]:
# 13. Связать "ДБсПризнакамиАртикула" с "Признаки для дат"
try:
    print("Начинаем создавать таблицу ДБсПризнаками...")
    start_time = time.time()
    df_final_db_all_features = pd.merge(df_final_db_item_features, df_date_features, on="Дата", how="left")
    df_final_db_all_features = format_date_column(df_final_db_all_features, 'Дата')

    print("Первые 5 строк таблицы ДБсПризнаками:")
    print(df_final_db_all_features.head())

    # Сохранение финальной таблицы
    # df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

    elapsed_time = time.time() - start_time
    print(f"Таблица ДБсПризнаками успешно создана. Время выполнения: {format_elapsed_time(elapsed_time)}")
except Exception as e:
    print(f"Ошибка при создании таблицы ДБсПризнаками: {e}")

Начинаем создавать таблицу ДБсПризнаками...
Первые 5 строк таблицы ДБсПризнаками:
         Дата   Артикул Две последние коллекции_x                Сезон_x  \
0  2025-09-03  00206040             2023SS,2024SS        лето (закрытое)   
1  2025-09-03  00206110             2023SS,2024SS        лето (закрытое)   
2  2025-09-03  00708270             2023AW,2024AW  демисезонный (низкий)   
3  2025-09-03  01908120             2022SS,2023SS        лето (открытое)   
4  2025-09-03  02908010             2024SS,2025SS  демисезонный (низкий)   

  Коллекция_x                Бренд_x  \
0      2024SS             T.TACCARDI   
1      2024SS             T.TACCARDI   
2      2024AW             T.TACCARDI   
3      2023SS  Alessio Nesca Comfort   
4      2025SS               TimeJump   

                                       Группа_x Розничный отдел_x  \
0                       002 Кеды женские летние     Женская обувь   
1                       002 Кеды женские летние     Женская обувь   
2  007 Полубо

In [50]:
import numpy as np
# df — ваша широкая таблица с флагами типов (1/0)
type_order_paid = ['Вывод в топ', 'Трафарет', 'Оплата за заказ']
paid_cols = [c for c in type_order_paid if c in df_final_db_all_features.columns]  # на случай отсутствующих

# Матрица флагов платных типов
flags = df_final_db_all_features[paid_cols].fillna(0).astype('uint8').to_numpy()
labels = np.array(paid_cols, dtype=object)

# Собираем подписи для платных комбинаций
combo = ['/'.join(labels[row.astype(bool)]) if row.any() else '' for row in flags]
df_final_db_all_features['ТипАктивности'] = combo

# Если есть только органика — подставим "Органика"
if 'Органика' in df_final_db_all_features.columns:
    only_org = df_final_db_all_features['Органика'].fillna(0).astype('uint8').eq(1) & (flags.sum(axis=1) == 0)
    df_final_db_all_features.loc[only_org, 'ТипАктивности'] = 'Органика'

# Пустые — на "—"
df_final_db_all_features['ТипАктивности'] = df_final_db_all_features['ТипАктивности'].replace('', '—')

In [51]:
# === SQL СЦЕПКИ ОЗОН ===
sql = """
SELECT scepka.[id]
      ,scepka.[offer_id]
      ,scepka.[product_id]
      ,sku.fbo_sku as [Артикул OZ]
      ,scepka.[group_value] as [Текущая склейка]
      ,sku.[article]
      ,scepka.[updated_at] as [Дата Обновления]
  FROM [DBReport].[mp].[ozon_scepka] scepka
  JOIN [DBReport].[mp].[ozon_sku] sku 
  ON  scepka.[product_id] = sku.[product_id] 
  and sku.actual = 1
"""
df_links = pd.read_sql(sql, engine)
df_links['Артикул OZ'] = df_links['Артикул OZ'].astype(str)
df_links.to_excel(os.path.join(FOLDER_PATH, f"Склейки товаров\\OZ\\{df_links['Дата Обновления'].iloc[0].strftime('%d.%m.%Y')}_Склейка Товаров_OZ.xlsx"))

In [52]:
count = 0
for item in list(df_final_db_all_features.columns):
    print(f'{{"{item}", type {str(type(df_final_db_all_features[item].unique()[0]))}}}, ', end="")
    count +=1
    if count == 5:
        print("\n", end="")
        count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"Две последние коллекции_x", type <class 'str'>}, {"Сезон_x", type <class 'str'>}, {"Коллекция_x", type <class 'str'>}, 
{"Бренд_x", type <class 'str'>}, {"Группа_x", type <class 'str'>}, {"Розничный отдел_x", type <class 'str'>}, {"Направление_x", type <class 'str'>}, {"Бизнес-группа_x", type <class 'str'>}, 
{"Себестоимость с НДС_x", type <class 'numpy.float64'>}, {"Процент выкупа_x", type <class 'numpy.float64'>}, {"Продажи, ₽", type <class 'NoneType'>}, {"Заказы, шт", type <class 'numpy.float64'>}, {"Показы", type <class 'numpy.float64'>}, 
{"Клики", type <class 'numpy.float64'>}, {"Корзины", type <class 'numpy.float64'>}, {"СтатусКоллекции", type <class 'str'>}, {"Вид товара", type <class 'str'>}, {"Артикул OZ", type <class 'str'>}, 
{"Наименование", type <class 'str'>}, {"Коллекция_y", type <class 'str'>}, {"Бренд_y", type <class 'str'>}, {"Сезон_y", type <class 'str'>}, {"Направление_y", type <class 'str'>}, 
{"Розн

In [ ]:
# # Шаг 1. Определим, какие колонки содержат суффиксы
# cols_x = [c for c in df_final_db_all_features.columns if c.endswith('_x')]
# cols_y = [c for c in df_final_db_all_features.columns if c.endswith('_y')]

# # Шаг 2. Уберем "_x" из названий колонок
# rename_map = {c: c[:-2] for c in cols_x}

# # Шаг 3. Выберем все нужные колонки:
# # - те, у которых есть "_x"
# # - все остальные, не имеющие "_y" и "_x"
# cols_final = [c for c in df_final_db_all_features.columns if not c.endswith('_y')]
# df_final_db_all_features_cutted = df_final_db_all_features[cols_final].rename(columns=rename_map)

In [ ]:
print(len(list(df_final_db_all_features.columns)))

118


In [ ]:
df_final_db_all_features

,Дата,Артикул,Две последние коллекции,Сезон,Коллекция,Бренд,Группа,Розничный отдел,Направление,Бизнес-группа,...,Признак Артикула 2,Признак Артикула 3,Признак Артикула 4,Признак Артикула 5,Признак Даты 1,Признак Даты 2,Признак Даты 3,Признак Даты 4,Признак Даты 5,ТипАктивности
0,2025-09-03,00206040,"2023SS,2024SS",лето (закрытое),2024SS,T.TACCARDI,002 Кеды женские летние,Женская обувь,Женская обувь,Обувь,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
1,2025-09-03,00206110,"2023SS,2024SS",лето (закрытое),2024SS,T.TACCARDI,002 Кеды женские летние,Женская обувь,Женская обувь,Обувь,...,NaN,ТОП 10% худшие,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
2,2025-09-03,00708270,"2023AW,2024AW",демисезонный (низкий),2024AW,T.TACCARDI,007 Полуботинки женские для активного отдыха,Женская обувь,Женская обувь,Обувь,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
3,2025-09-03,01908120,"2022SS,2023SS",лето (открытое),2023SS,Alessio Nesca Comfort,019 Сандалии женские,Женская обувь,Женская обувь,Обувь,...,NaN,ТОП 10% худшие,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
4,2025-09-03,02908010,"2024SS,2025SS",демисезонный (низкий),2025SS,TimeJump,029 Полуботинки мужские для активного отдыха,Мужская обувь,Мужская обувь,Обувь,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329453,2025-10-14,y5608030,2025SS,демисезонный,2025SS,Kari KIDS,y56 Куртка для девочек облегченная,Верхняя детская одежда для девочек,Одежда детская для девочек,Одежда для детей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
329454,2025-10-14,y5608060,2025SS,демисезонный,2025SS,kari,y56 Куртка для девочек облегченная,Верхняя детская одежда для девочек,Одежда детская для девочек,Одежда для детей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
329455,2025-10-14,y5908000,2025SS,демисезонный,2025SS,Kari baby,y59 Ветровка для маленьких мальчиков,Одежда для новорожденных,Одежда для новорожденных,Одежда для детей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет
329456,2025-10-14,y5908030,2025SS,демисезонный,2025SS,Kari baby,y59 Ветровка для маленьких мальчиков,Одежда для новорожденных,Одежда для новорожденных,Одежда для детей,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Трафарет


In [ ]:
# count = 0
# for item in list(df_final_db_all_features_cutted.columns):
#     print(f'{{"{item}", type {str(type(df_final_db_all_features_cutted[item].unique()[0]))}}}, ', end="")
#     count +=1
#     if count == 5:
#         print("\n", end="")
#         count = 0

{"Дата", type <class 'str'>}, {"Артикул", type <class 'str'>}, {"Две последние коллекции", type <class 'str'>}, {"Сезон", type <class 'str'>}, {"Коллекция", type <class 'str'>}, 
{"Бренд", type <class 'str'>}, {"Группа", type <class 'str'>}, {"Розничный отдел", type <class 'str'>}, {"Направление", type <class 'str'>}, {"Бизнес-группа", type <class 'str'>}, 
{"Себестоимость с НДС", type <class 'numpy.float64'>}, {"Процент выкупа", type <class 'numpy.float64'>}, {"Продажи, ₽", type <class 'NoneType'>}, {"Заказы, шт", type <class 'numpy.float64'>}, {"Показы", type <class 'numpy.float64'>}, 
{"Клики", type <class 'numpy.float64'>}, {"Корзины", type <class 'numpy.float64'>}, {"СтатусКоллекции", type <class 'str'>}, {"Вид товара", type <class 'str'>}, {"Артикул OZ", type <class 'str'>}, 
{"Наименование", type <class 'str'>}, {"Модель", type <class 'str'>}, {"Техсегмент", type <class 'str'>}, {"Байер", type <class 'str'>}, {"Основной артикул", type <class 'str'>}, 
{"НДС", type <class 'numpy.

In [ ]:
import pyarrow as pa
import pyarrow.csv as csv
table = pa.Table.from_pandas(df_final_db_all_features)

In [ ]:
csv.write_csv(table, os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"))

In [ ]:
# Сохранение финальной таблицы
# df_final_db_all_features.to_csv(os.path.join(FOLDER_PATH, "ДБсПризнаками_Ozon_New.csv"), index=False)

In [ ]:
# del df_date_features
# gc.collect()

In [ ]:
# Функция для обновления Excel-файла с циклом попыток
def update_and_save_excel(file_path, new_file_path):
    max_attempts = 10  # Максимальное количество попыток
    attempt = 0

    while attempt < max_attempts:
        attempt += 1
        print(f"Попытка {attempt} обновить файл '{os.path.basename(file_path)}'...")

        try:
            # Открываем Excel приложение
            excel = win32.Dispatch("Excel.Application")
            excel.DisplayAlerts = False  # Отключает предупреждения Excel

            try:
                # Открываем книгу
                workbook = excel.Workbooks.Open(file_path)

                # Выполняем обновление всех данных (эквивалентно "Обновить всё" в Excel)
                print("Выполняем обновление данных...")
                workbook.RefreshAll()
                excel.CalculateUntilAsyncQueriesDone()  # Дожидаемся завершения обновления

                # Сохраняем оригинальный файл в FOLDER_PATH_FOR_DB
                workbook.SaveAs(file_path)
                print(f"Файл успешно сохранен с оригинальным именем в '{os.path.dirname(file_path)}'.")

                # Сохраняем файл с новым именем в FOLDER_PATH_FEATURES
                workbook.SaveAs(new_file_path)
                print(f"Файл успешно сохранен как '{os.path.basename(new_file_path)}'.")

                return True  # Успешное завершение

            except Exception as e:
                print(f"Ошибка при обновлении или сохранении файла: {e}")
            finally:
                # Закрываем книгу и выходим из Excel
                if 'workbook' in locals():
                    workbook.Close(SaveChanges=False)
                excel.Quit()

        except Exception as e:
            print(f"Ошибка при работе с Excel: {e}")

        # Если произошла ошибка, ждем перед следующей попыткой
        if attempt < max_attempts:
            print(f"Пауза перед следующей попыткой ({attempt + 1}/{max_attempts})...")
            time.sleep(60)  # Пауза 5 секунд

    return False  # Все попытки завершились неудачно

In [ ]:
# 19. Обновить файл "Показы и затраты ОЗ_2.0.xlsx"
try:
    print("Подготовка данных для ДБ завершена.")
    # input("Начать обновление файлов ДБ? Для подтверждения нажмите Enter...")
    print("Начинаем обновлять файл 'Показы и затраты ОЗ_2.0.xlsx'...")
    start_time = time.time()  # Запускаем таймер

    # Путь к исходному файлу
    file_path_shows_expenses = os.path.join(FOLDER_PATH_FOR_DB, "Показы и затраты ОЗ_2.0_New.xlsx")

    if os.path.exists(file_path_shows_expenses):
        # Создаем новое имя файла с текущей датой без года
        current_month_day = time.strftime("%d.%m")  # Текущая дата в формате ДД.ММ
        new_file_name = f"Показы и затраты ОЗ_2.0 {current_month_day}_New.xlsx"
        new_file_path = os.path.join(FOLDER_PATH_FEATURES, new_file_name)

        # Путь для сохранения в дополнительную папку FOLDER_PATH_DUDL
        dudl_file_path = os.path.join(FOLDER_PATH_DUDL, new_file_name)

        # Удаляем старые файлы из FOLDER_PATH_DUDL
        try:
            if os.path.exists(FOLDER_PATH_DUDL):
                for filename in os.listdir(FOLDER_PATH_DUDL):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_New.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_DUDL, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_DUDL}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_DUDL}': {delete_error}")

        # Удаляем старые файлы из FOLDER_PATH_FEATURES
        try:
            if os.path.exists(FOLDER_PATH_FEATURES):
                for filename in os.listdir(FOLDER_PATH_FEATURES):
                    # Ищем файлы с шаблоном "Показы и затраты ОЗ_2.0 DD.MM.xlsx"
                    match = re.match(r"Показы и затраты ОЗ_2\.0 (\d{2}\.\d{2})\_New.xlsx", filename)
                    if match:
                        file_date = match.group(1)  # Извлекаем дату из имени файла
                        if file_date != current_month_day:  # Сравниваем с текущей датой
                            file_to_delete = os.path.join(FOLDER_PATH_FEATURES, filename)
                            os.remove(file_to_delete)
                            print(f"Файл '{filename}' удален из папки '{FOLDER_PATH_FEATURES}'.")
        except Exception as delete_error:
            print(f"Ошибка при удалении старых файлов из папки '{FOLDER_PATH_FEATURES}': {delete_error}")

        # Пытаемся обновить и сохранить файл
        success = update_and_save_excel(file_path_shows_expenses, new_file_path)

        if not success:
            # Если все попытки неудачны, выводим сообщение пользователю
            while not success:
                input("Обновить Excel файл не получилось. Закройте все открытые файлы и нажмите любую кнопку для повторной попытки.")
                success = update_and_save_excel(file_path_shows_expenses, new_file_path)

            print("Файл успешно обновлен после повторной попытки.")

        # После успешного обновления копируем файл в папку FOLDER_PATH_DUDL
        if success:
            try:
                shutil.copy(new_file_path, dudl_file_path)
                print(f"Файл успешно скопирован в папку '{FOLDER_PATH_DUDL}'.")
            except Exception as copy_error:
                print(f"Ошибка при копировании файла в папку '{FOLDER_PATH_DUDL}': {copy_error}")

        elapsed_time = time.time() - start_time  # Вычисляем затраченное время
        print(f"Файл успешно обновлен и сохранен. Время выполнения: {format_elapsed_time(elapsed_time)}")
    else:
        print("Файл 'Показы и затраты ОЗ_2.0.xlsx' не найден.")
except Exception as e:
    print(f"Ошибка при обработке файла 'Показы и затраты ОЗ_2.0.xlsx': {e}")